# LLM judges on the same prompts

The typed decision models were compared with CALM's published numbers for its LLM judges, which ran on CALM's prompts at
temperature 0.7 (Section 4.2). Here LLM judges answer exactly our prompts, at temperature 0, so that both kinds of judge
are measured the same way: two of CALM's 2024 judges that are still served, and current (2026) LLMs, among them
Qwen3.8-27B, the base LLM of Perplexity Decider and Clef. Hidden reasoning ("thinking") is switched off where the provider allows
it, since CALM's prompt already asks for a written explanation before the verdict.

**SMOKE = True** sends a few prompts per test to every judge, to check that each model answers in CALM's format and to
estimate the cost and time of the full design (14,695 requests per judge). Set it to False for the full runs.

In [ ]:
SMOKE = False
PER_TEST = 3          # smoke run: prompts per test (and of the originals)

In [ ]:
import sys
from pathlib import Path
CODE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE))   # judgebias and tdm
import pandas as pd
from judgebias import authority, design, llm_judges as L
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
rows = design.build_design(citations=authority.load_citations())
pd.DataFrame([{"label": k, "model": m, "generation": g, "reasoning": r} for k, (m, g, r) in L.MODELS.items()])

,label,model,generation,reasoning
0,ChatGPT,openai/gpt-3.5-turbo,2024,{'enabled': False}
1,GPT-4o,openai/gpt-4o-2024-05-13,2024,{'enabled': False}
2,GPT-6.1 Sol,openai/gpt-6.1-sol,2026,{'effort': 'low'}
3,Claude Sonnet 5.5,anthropic/claude-sonnet-5.5,2026,{'effort': 'low'}
4,Gemini 3.8 Flash,google/gemini-3.8-flash,2026,{'effort': 'low'}
5,Qwen3.8-27B,qwen/qwen3.8-27b,2026,{'enabled': False}
6,Claude Haiku 5.5,anthropic/claude-haiku-5.5,2026,{'enabled': False}


## Smoke run: does every model answer, and what would the full design cost?

Per model: share of replies with a verdict ([[A]] or [[B]]), mean tokens, cost and median latency in the sample, and the
extrapolation to the 14,695 requests of the full design. The time assumes 16 or 32 requests in parallel and no rate
limits, so it is a lower bound.

In [ ]:
picked = L.smoke_sample(rows, per_test=PER_TEST)
print(len(picked), "prompts per model")
est = []
for label, (model, gen, reasoning) in L.MODELS.items():
    recs = L.run(model, picked, smoke=True, concurrency=8, budget_usd=2, reasoning=reasoning)
    est.append({"label": label, "generation": gen, **L.estimate(recs)})
est = pd.DataFrame(est)
est.to_csv(CODE / "outputs" / "tables" / "llm_judges_smoke.csv", index=False)
est.round(3)

30 prompts per model


openai/gpt-3.5-turbo: 100%|##########| 30/30 [00:00<?, ?it/s]

openai/gpt-4o-2024-05-13: 100%|##########| 30/30 [00:00<?, ?it/s]

openai/gpt-6.1-sol:   0%|          | 0/30 [00:00<?, ?it/s]

anthropic/claude-sonnet-5.5:   0%|          | 0/30 [00:00<?, ?it/s]

google/gemini-3.8-flash:   0%|          | 0/30 [00:00<?, ?it/s]

qwen/qwen3.8-27b: 100%|##########| 30/30 [00:00<?, ?it/s]

anthropic/claude-haiku-5.5: 100%|##########| 30/30 [00:00<?, ?it/s]

,label,generation,sample,ok,verdict_found,input_tokens,output_tokens,reasoning_tokens,usd_sample,usd_per_1000,usd_full_design,median_latency_s,hours_full_at_16,hours_full_at_32
0,ChatGPT,2024,30,30,29,784.267,185.167,0.000,0.020,0.670,9.844,1.763,0.484,0.242
1,GPT-4o,2024,30,30,30,775.767,230.700,0.000,0.220,7.339,107.852,2.764,0.677,0.339
2,GPT-6.1 Sol,2026,30,30,30,774.767,88.133,7.267,0.076,2.535,37.255,3.373,0.946,0.473
3,Claude Sonnet 5.5,2026,30,30,30,1184.400,345.633,0.000,0.175,5.825,85.600,3.840,0.967,0.484
4,Gemini 3.8 Flash,2026,30,30,30,810.300,241.133,135.667,0.045,1.512,22.218,3.458,1.240,0.620
5,Qwen3.8-27B,2026,30,30,30,816.300,341.100,0.000,0.027,0.893,13.122,6.346,2.145,1.073
6,Claude Haiku 5.5,2026,30,30,30,1182.400,386.567,0.000,0.009,0.312,4.578,2.917,0.741,0.370


In [ ]:
tot = est[["usd_sample", "usd_full_design", "hours_full_at_16", "hours_full_at_32"]].sum()
print(f"smoke run cost US${tot.usd_sample:.2f}; full design, all models: US${tot.usd_full_design:.0f}, "
      f"{tot.hours_full_at_16:.1f} h at 16 parallel or {tot.hours_full_at_32:.1f} h at 32 (one model after another)")
fails = [(r["model"], r["error"]) for label, (m, g, r) in L.MODELS.items() for r in L.load_records(m, smoke=True) if not r["ok"]]
print("failed requests:", len(fails)); fails[:10]

smoke run cost US$0.57; full design, all models: US$280, 7.2 h at 16 parallel or 3.6 h at 32 (one model after another)
failed requests: 0


[]

## Full runs

Every judge answers the full design (14,695 requests) at 16 requests in parallel, with the key in `OPENROUTER_API_KEY`.
The judges run in the order below, each with a budget of about 1.4 times its estimate from the smoke run; the last
ones get what is left on the key. A run that stops (budget, key limit, crash) is continued by running this cell again:
answers already received come from the cache and cost nothing, only the missing requests are sent. Claude Sonnet 5.5
runs on Anthropic's API instead (next section).

In [ ]:
ORDER = [("ChatGPT", 14), ("Qwen3.8-27B", 16), ("GPT-6.1 Sol", 52), ("Gemini 3.8 Flash", 30),
         ("Claude Haiku 5.5", 7), ("GPT-4o", 150)]       # (judge, budget in US$); Claude Sonnet 5.5 runs below
if not SMOKE:
    done = []
    for label, budget in ORDER:
        left = L.key_status()["limit_remaining"]
        if left is not None and left < 1:
            print(f"key limit reached, stopping before {label}"); break
        recs = L.run_full(label, rows, budget_usd=min(budget, left - 0.5) if left is not None else budget,
                          concurrency=16)
        ok = sum(r["ok"] for r in recs)
        done.append({"label": label, "requests": len(recs), "answered": ok, "missing": len(recs) - ok,
                     "verdict_found": sum(r["verdict"] is not None for r in recs if r["ok"]),
                     "usd": round(sum(r["cost"] or 0 for r in recs if r["ok"] and not r["cached"]), 2),
                     "key_left": L.key_status()["limit_remaining"]})
        print(done[-1])
    display(pd.DataFrame(done))

openai/gpt-3.5-turbo: 100%|##########| 14695/14695 [00:00<?, ?it/s]

{'label': 'ChatGPT', 'requests': 14695, 'answered': 14695, 'missing': 0, 'verdict_found': 14604, 'usd': 0, 'key_left': 103.684459411}


qwen/qwen3.8-27b: 100%|##########| 14695/14695 [00:00<?, ?it/s]

{'label': 'Qwen3.8-27B', 'requests': 14695, 'answered': 14695, 'missing': 0, 'verdict_found': 14680, 'usd': 0, 'key_left': 103.684459411}


openai/gpt-6.1-sol: 100%|##########| 14695/14695 [00:00<?, ?it/s]

...

anthropic/claude-haiku-5.5: 100%|##########| 14695/14695 [00:00<?, ?it/s]

{'label': 'Claude Haiku 5.5', 'requests': 14695, 'answered': 14695, 'missing': 0, 'verdict_found': 14402, 'usd': 0, 'key_left': 103.660034411}


openai/gpt-4o-2024-05-13: 100%|##########| 14695/14695 [00:00<?, ?it/s]

{'label': 'GPT-4o', 'requests': 14695, 'answered': 14695, 'missing': 0, 'verdict_found': 14686, 'usd': 0, 'key_left': 103.660034411}


,label,requests,answered,missing,verdict_found,usd,key_left
0,ChatGPT,14695,14695,0,14604,0,103.684459
1,Qwen3.8-27B,14695,14695,0,14680,0,103.684459
2,GPT-6.1 Sol,14695,14695,0,14695,0,103.684459
3,Gemini 3.8 Flash,14695,14651,44,14631,0,103.660034
4,Claude Haiku 5.5,14695,14695,0,14402,0,103.660034
5,GPT-4o,14695,14695,0,14686,0,103.660034


## Claude Sonnet 5.5 directly on Anthropic's API

Sonnet runs directly on Anthropic's API (key in `ANTHROPIC_API_KEY`). Thinking is off
(`between_tools`); the model accepts no temperature. Every answer is saved as it arrives, and running the cell again
only sends what is missing. Anthropic's Batch API would halve the cost but answers within up to 24 hours; a test batch
of 30 requests sat unprocessed for 7 minutes and was cancelled (`L.batch_submit` and `L.batch_collect` remain in the
module). **SONNET_SMOKE = True** sends the 30 prompts of the smoke run.

In [ ]:
SONNET_SMOKE = False
label = "Claude Sonnet 5.5"
if SONNET_SMOKE:
    recs = L.run_anthropic(label, L.smoke_sample(rows, per_test=PER_TEST), smoke=True)
else:
    reqs = L.design_run.requests(rows, None)
    recs = L.run_anthropic(label, [r for r, _ in reqs], [k for _, k in reqs], concurrency=32)
pd.DataFrame([{"label": label, **L.estimate(recs)}]).round(3)

,label,sample,ok,verdict_found,input_tokens,output_tokens,reasoning_tokens,usd_sample,usd_per_1000,usd_full_design,median_latency_s,hours_full_at_16,hours_full_at_32
0,Claude Sonnet 5.5,14695,14695,14695,1202.504,488.97,0.0,107.196,7.295,107.196,4.795,1.41,0.705


## Notes

**First smoke run, 2026-10-08, with thinking and GLM-5.3** (results in `outputs/tables/llm_judges_smoke_with_thinking.csv`) (30 prompts per model, 3 per test, temperature 0, 8 in parallel). All 240 requests answered;
239 replies end with a verdict (one ChatGPT reply has none). The sample cost US$0.83 in total.

* Extrapolated to the full design (14,695 requests per model): about US$411 for all eight models. GPT-4o (US$108),
  Claude Sonnet 5.5 (US$91) and GLM-5.3 (US$90) are the expensive ones; ChatGPT (US$10) and Claude Haiku 5.5 (US$7)
  the cheapest.
* Four models think before they answer (reasoning tokens per reply: GLM-5.3 about 1,700, Qwen3.8-27B 900, Haiku and
  Gemini about 300, GPT-6.1 Sol 20). This drives their cost and time. CALM's protocol asks for a written explanation,
  so the hidden reasoning could be switched off where the provider allows it.
* Time: median 2 to 10 s per request. Qwen3.8-27B and GLM-5.3 have long tails (up to about 3.5 minutes per request),
  which made the smoke run of Qwen take 10 minutes. With 32 requests in parallel and all models running at the same
  time, the full design takes about 2.5 hours (Qwen and GLM), the others under an hour.
* Claude models count about 1,180 input tokens per prompt (other tokenizer), the others about 780 to 860.

**Second smoke run, 2026-10-08, seven models, thinking off where possible** (`outputs/tables/llm_judges_smoke.csv`). GLM-5.3
dropped. Thinking switched off for Qwen3.8-27B and Claude Haiku 5.5 (0 reasoning tokens) and for the two 2024 models,
which have none. GPT-6.1 Sol, Claude Sonnet 5.5 and Gemini 3.8 Flash refuse ("Reasoning is mandatory for this endpoint
and cannot be disabled") and run at low effort (7, 0 and 136 reasoning tokens per reply). All 210 requests answered,
209 with a verdict. Full design: about US$280 for the seven models (US$321 with thinking), the slowest model
(Qwen3.8-27B) about 1.1 hours at 32 requests in parallel, so about an hour if all models run at the same time.

**2026-10-09 evening:** the design grew by 500 prompts (our padded better responses, verbosity on the better response), to 15,195 requests per judge. All seven judges answered them, sent with the functions of `judgebias/llm_judges.py` (only the new prompts); Gemini again returns empty replies for the 44 role-play prompts.